# Rigorous Offline AIRL Pipeline
### Replicating the 'Bridging the Gap' Methodology
This notebook provides a robust, academic-grade implementation of Adversarial Inverse Reinforcement Learning (AIRL). 
Instead of a quick approximation, this pipeline takes offline CSV data (recorded from your Unreal Engine game), feeds it into a deep learning discriminator, and optimizes a Generator policy to match it. 

Ultimately, it isolates the `g_theta` (pure reward preferences) from `h_phi` (environment dynamics).

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy

from imitation.algorithms.adversarial.airl import AIRL
from imitation.data import types
from imitation.rewards.reward_nets import BasicShapedRewardNet
from imitation.util.networks import RunningNorm
from imitation.util.util import make_vec_env

c:\Users\Danie\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\gym\envs\registration.py:307: DeprecationWarning: The package name gym_minigrid has been deprecated in favor of minigrid. Please uninstall gym_minigrid and install minigrid with `pip install minigrid`. Future releases will be maintained under the new package name minigrid.
  fn()
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 1. The Offline Environment Wrapper
Even though we are training offline from CSVs, the RL Generator still needs an environment to interact with so it can learn its policy and generate 'fake' trajectories to fool the discriminator.

In [2]:
from gymnasium.envs.registration import register

class OfflineDrivingEnv(gym.Env):
    """
    A mathematical representation of your Unreal Engine driving task.
    This allows the Generator agent to explore the state space.
    """
    def __init__(self):
        super().__init__()
        # Action: [Steering (-1 to 1), Throttle (0 to 1)]
        self.action_space = spaces.Box(low=np.array([-1.0, 0.0]), high=np.array([1.0, 1.0]), dtype=np.float32)
        
        # State: [Normalized Speed, Normalized Dist, Normalized Dev]
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(3,), dtype=np.float32)
        
        # Mock internal state for the generator to step through
        self.current_step = 0
        self.max_steps = 1000

    def step(self, action):
        self.current_step += 1
        # The generator creates a mock state based on its action (simplified physics)
        mock_state = np.random.uniform(0, 1, size=(3,)).astype(np.float32)
        
        reward = 0.0 # IRL assumes reward is 0, we are trying to learn it!
        done = self.current_step >= self.max_steps
        return mock_state, reward, done, False, {}

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mock_state = np.array([0.5, 0.5, 0.5], dtype=np.float32)
        return mock_state, {}

# 1. Register the custom environment with Gymnasium
register(
    id='OfflineDriving-v0',
    entry_point='__main__:OfflineDrivingEnv',
)

# 2. Vectorize using the registered string ID
venv = make_vec_env('OfflineDriving-v0', n_envs=1, rng=np.random.default_rng(42))

## 2. Loading Human Trajectories from CSV
In academic research, you record the player's sessions to CSV files. Here, we simulate loading a real human's driving data and parsing it into the `imitation` library's Trajectory format.

In [3]:
def load_human_data_from_csv(filepath="mock_data.csv"):
    """
    In reality, you would use pandas to read the CSV: 
    df = pd.read_csv(filepath)
    obs = df[['speed', 'dist', 'dev']].values
    acts = df[['steering', 'throttle']].values
    """
    print("Loading human trajectory data...")
    
    # MOCKING THE DATA LOAD:
    # 5 episodes, 1000 steps each
    trajectories = []
    for _ in range(5):
        # Human states (N+1 length)
        obs = np.random.uniform(0.2, 0.8, size=(1001, 3)).astype(np.float32)
        # Human actions (N length)
        acts = np.random.uniform(0, 1, size=(1000, 2)).astype(np.float32)
        infos = [{}] * 1000
        
        traj = types.Trajectory(obs=obs, acts=acts, infos=infos, terminal=True)
        trajectories.append(traj)
        
    print(f"Loaded {len(trajectories)} expert trajectories.")
    return trajectories

expert_trajectories = load_human_data_from_csv()

Loading human trajectory data...
Loaded 5 expert trajectories.


## 3. ...
...

In [4]:
import os
from imitation.util.logger import configure

generator_path = "airl_generator_model.zip"
reward_path = "airl_reward_net.pth"

# 1. Initialize the base networks
generator_agent = PPO(
    MlpPolicy, 
    venv, 
    verbose=0, 
    n_steps=2048, 
    ent_coef=0.01
)

reward_net = BasicShapedRewardNet(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    normalize_input_layer=RunningNorm,
    reward_hid_sizes=[],
    potential_hid_sizes=[32, 32]
)

airl_trainer = AIRL(
    demonstrations=expert_trajectories,
    demo_batch_size=1024,
    gen_replay_buffer_capacity=2048,
    n_disc_updates_per_round=4,
    venv=venv,
    gen_algo=generator_agent,
    reward_net=reward_net,
    # REMOVED allow_variable_horizon=True to fix the unexpected keyword error!
    # Our custom environment has a fixed 1000-step limit, so it is no longer needed.
)

# 2. Check for existing models to Load OR Train
if os.path.exists(generator_path) and os.path.exists(reward_path):
    print("Existing models found! Loading from disk...")
    generator_agent = PPO.load(generator_path, env=venv)
    airl_trainer.gen_algo = generator_agent
    reward_net.load_state_dict(torch.load(reward_path))
    print("Models successfully loaded!")
else:
    print("No existing models found. Commencing Rigorous AIRL Training...")
    custom_logger = configure(folder="airl_logs", format_strs=["stdout", "tensorboard"])
    airl_trainer.logger = custom_logger
    generator_agent.set_logger(custom_logger)
    
    airl_trainer.train(total_timesteps=20480)
    print("Training Complete!\n")
    
    print("Saving the models...")
    generator_agent.save(generator_path)
    torch.save(reward_net.state_dict(), reward_path)
    print(f"Models saved successfully to disk as '{generator_path}' and '{reward_path}'!")


No existing models found. Commencing Rigorous AIRL Training...


round:   0%|          | 0/10 [00:00<?, ?it/s]

------------------------------------------
| raw/                        |          |
|    gen/rollout/ep_len_mean  | 1e+03    |
|    gen/rollout/ep_rew_mean  | 0        |
|    gen/time/fps             | 393      |
|    gen/time/iterations      | 1        |
|    gen/time/time_elapsed    | 5        |
|    gen/time/total_timesteps | 2048     |
------------------------------------------
--------------------------------------------------
| raw/                                |          |
|    disc/disc_acc                    | 0.5      |
|    disc/disc_acc_expert             | 1        |
|    disc/disc_acc_gen                | 0        |
|    disc/disc_entropy                | 0.271    |
|    disc/disc_loss                   | 1.29     |
|    disc/disc_proportion_expert_pred | 1        |
|    disc/disc_proportion_expert_true | 0.5      |
|    disc/global_step                 | 1        |
|    disc/n_expert                    | 1.02e+03 |
|    disc/n_generated                 | 1.02e+03 |
-

round:  10%|█         | 1/10 [00:06<01:02,  6.99s/it]

-----------------------------------------------------
| raw/                               |              |
|    gen/rollout/ep_len_mean         | 1e+03        |
|    gen/rollout/ep_rew_mean         | 0            |
|    gen/rollout/ep_rew_wrapped_mean | 295          |
|    gen/time/fps                    | 559          |
|    gen/time/iterations             | 1            |
|    gen/time/time_elapsed           | 3            |
|    gen/time/total_timesteps        | 4096         |
|    gen/train/approx_kl             | 0.01950714   |
|    gen/train/clip_fraction         | 0.238        |
|    gen/train/clip_range            | 0.2          |
|    gen/train/entropy_loss          | -2.83        |
|    gen/train/explained_variance    | -0.016257405 |
|    gen/train/learning_rate         | 0.0003       |
|    gen/train/loss                  | 0.672        |
|    gen/train/n_updates             | 10           |
|    gen/train/policy_gradient_loss  | -0.0283      |
|    gen/train/std          

...

-------------------------------------------------------
| raw/                               |                |
|    gen/rollout/ep_len_mean         | 1e+03          |
|    gen/rollout/ep_rew_mean         | 0              |
|    gen/rollout/ep_rew_wrapped_mean | 448            |
|    gen/time/fps                    | 529            |
|    gen/time/iterations             | 1              |
|    gen/time/time_elapsed           | 3              |
|    gen/time/total_timesteps        | 18432          |
|    gen/train/approx_kl             | 0.0182827      |
|    gen/train/clip_fraction         | 0.167          |
|    gen/train/clip_range            | 0.2            |
|    gen/train/entropy_loss          | -2.74          |
|    gen/train/explained_variance    | -0.00023245811 |
|    gen/train/learning_rate         | 0.0003         |
|    gen/train/loss                  | 1.64           |
|    gen/train/n_updates             | 80             |
|    gen/train/policy_gradient_loss  | -0.0155  

round:  90%|█████████ | 9/10 [00:55<00:06,  6.13s/it]

-------------------------------------------------------
| raw/                               |                |
|    gen/rollout/ep_len_mean         | 1e+03          |
|    gen/rollout/ep_rew_mean         | 0              |
|    gen/rollout/ep_rew_wrapped_mean | 472            |
|    gen/time/fps                    | 508            |
|    gen/time/iterations             | 1              |
|    gen/time/time_elapsed           | 4              |
|    gen/time/total_timesteps        | 20480          |
|    gen/train/approx_kl             | 0.007189596    |
|    gen/train/clip_fraction         | 0.0989         |
|    gen/train/clip_range            | 0.2            |
|    gen/train/entropy_loss          | -2.71          |
|    gen/train/explained_variance    | -0.00038528442 |
|    gen/train/learning_rate         | 0.0003         |
|    gen/train/loss                  | 8.53           |
|    gen/train/n_updates             | 90             |
|    gen/train/policy_gradient_loss  | -0.00784 

round: 100%|██████████| 10/10 [01:00<00:00,  6.08s/it]

Training Complete!

Saving the models...
Models saved successfully to disk as 'airl_generator_model.zip' and 'airl_reward_net.pth'!


## 5. Extract Subjective Weights & Calculate Impulsivity
Because we specifically constrained `reward_kwargs={"hid_sizes": []}`, the base reward network is a simple linear layer. We can pluck the weights directly out of the PyTorch layer, just like standard linear regression!

In [5]:
import torch.nn as nn

# The base reward network is inside the reward_net
base_reward_model = reward_net.base

# Safely find the Linear layer (imitation often appends a SqueezeLayer at the end)
linear_layer = None
for module in base_reward_model.mlp.modules():
    if isinstance(module, nn.Linear):
        linear_layer = module
        break

# Now we safely extract the weights mapping directly to [Speed, Dist, Dev]
learned_weights = linear_layer.weight.detach().cpu().numpy()[0]

w_speed = learned_weights[0]
w_dist = learned_weights[1]
w_dev = learned_weights[2]

print("--- Extracted Human Subjective Preferences ---")
print(f"Value of Speed:          {w_speed:+.4f}")
print(f"Value of Safety Dist:    {w_dist:+.4f}")
print(f"Value of Lane Dev:       {w_dev:+.4f}\n")

# Your specific Impulsivity Math
def calculate_impulsivity_score(w_s, w_d):
    if abs(w_d) < 0.001: w_d = -0.001
    raw_score = w_s / abs(w_d)
    return min(max(raw_score * 50, 0), 100)

score = calculate_impulsivity_score(w_speed, w_dist)
print(f"FINAL PSYCHOLOGICAL IMPULSIVITY SCORE: {score:.2f} / 100")

--- Extracted Human Subjective Preferences ---
Value of Speed:          -0.2366
Value of Safety Dist:    -0.2437
Value of Lane Dev:       +0.0687

FINAL PSYCHOLOGICAL IMPULSIVITY SCORE: 0.00 / 100
